# 5 — Deploying an agentic workflow as an API

Week 3 Segment 4: **Deployment and Production Patterns.**

We've talked about evaluation (Segment 1), observability (Segment 2), and human-in-the-loop / guardrails (Segment 3). The exercise for this segment is:

> **Deploy an agent workflow as an API with health checks and monitoring.**

The workflow lives in [`apps/agent_api/`](../../apps/agent_api/) and is a real **deep-research loop** with agents in the loop:

```mermaid
flowchart LR
    start([POST /research<br/>topic]) --> plan["plan_node<br/>structured LLM<br/>emits ResearchPlan<br/>(2-5 sub-queries)"]
    plan --> agent["agent_node<br/>ReAct with SerpAPI + Firecrawl<br/>answers each sub-query<br/>with [url] citations"]
    agent --> reflect["reflect_node<br/>structured LLM<br/>done? or missing_questions?"]
    reflect --> gate{verdict.done<br/>or<br/>round >= max?}
    gate -- not done --> plan
    gate -- finalize --> artifact["artifact_node<br/>final cited markdown<br/>+ SQLite persist"]
    artifact --> done([artifact_id])
```

- A **structured-output LLM** turns the topic into a `ResearchPlan` — 2-5 focused, searchable sub-queries. On retry rounds it also reads the previous round's `missing_questions` so the next plan targets specific gaps.
- A **research agent** (`build_solo` from `src/multi_agent/topologies.py`) executes one sub-query at a time using **`serpapi_search`** (Google via SerpAPI) and **`firecrawl_scrape`** (Firecrawl's clean markdown extraction, with a naïve `httpx` fallback if no Firecrawl key). It returns 2-4 sentences per sub-query with inline `[url]` citations.
- A **reflect LLM** looks at the accumulated findings and returns `{done, reasoning, missing_questions}`. If not done, the workflow loops back into `plan` with those follow-ups; if done (or we've hit `max_iterations`), it falls through to the writer.
- An **artifact LLM** synthesizes the final cited markdown report from every round's findings and we persist to SQLite.

The deployment skin around that workflow is what Segment 4 cares about:

| concern | how the app handles it |
|---|---|
| liveness | `GET /healthz` — 200 if the process is alive |
| readiness | `GET /readyz` — 503 unless DB is writable AND the LLM key resolves AND SerpAPI is configured |
| monitoring | `GET /metrics` — Prometheus exposition with workflow / reflect / HTTP counters |
| structured logs | JSON-per-line, one record per event, `request_id` propagated via `ContextVar` |
| trace replay | `GET /trace?request_id=...` — tail the log filtered to one request |
| containerization | `Dockerfile` + `docker-compose.yml` with a `HEALTHCHECK` on `/readyz` |
| Forge handoff | FastMCP server under `apps/agent_api/mcp_server/` wraps the API as three tools |

This notebook is a walkthrough: bring the service up, curl every endpoint, exercise the iteration cap, then install the MCP wrapper into Forge and call the same workflow from a Forge chat turn.

## 1. Bring up the service

There are two equivalent ways to run the service for this notebook:

**Local uvicorn (fastest iteration loop):**

```bash
cd apps/agent_api
pip install -r requirements.txt
PYTHONPATH=../../src uvicorn app.main:app --port 8090 --reload
```

**Docker compose (closer to how you'd ship it):**

```bash
cd apps/agent_api
cp .env.example .env
# at minimum, set OPENROUTER_API_KEY and SERPAPI_API_KEY.
# FIRECRAWL_API_KEY is optional but strongly recommended — without it
# we fall back to a naive httpx GET + HTML strip that breaks on JS-heavy
# pages.
docker compose up --build
```

The compose file mounts the repo's `src/` read-only at `/app/src` so the workflow keeps using `from shared import get_llm`. The container's healthcheck calls `/readyz` every 10s, so `docker compose ps` will show `(healthy)` once both the LLM key and SerpAPI key are present.

The cell below starts a local uvicorn process for you if the port is free, then waits for `/healthz` to come up. Skip this cell if you started the service some other way.

In [1]:
import os, sys, json, subprocess, time
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))
load_dotenv(Path("../../apps/agent_api/.env"))

BASE = "http://localhost:8090"
APP_DIR = Path("../../apps/agent_api").resolve()
SRC_DIR = Path("../../src").resolve()

def _is_up(timeout=1.0) -> bool:
    try:
        return requests.get(f"{BASE}/healthz", timeout=timeout).ok
    except requests.RequestException:
        return False

if _is_up():
    print("agent_api already responding on :8090 — reusing it.")
    proc = None
else:
    env = {**os.environ, "PYTHONPATH": f"{SRC_DIR}:{APP_DIR}"}
    print("starting uvicorn in the background...")
    proc = subprocess.Popen(
        ["uvicorn", "app.main:app", "--port", "8090"],
        cwd=str(APP_DIR),
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    deadline = time.time() + 30
    while time.time() < deadline and not _is_up():
        time.sleep(0.5)
    if not _is_up():
        proc.terminate()
        raise RuntimeError("agent_api never came up; check the uvicorn logs.")
    print(f"agent_api ready at {BASE} (pid={proc.pid})")

agent_api already responding on :8090 — reusing it.


## 2. Health and readiness — what's the difference?

`/healthz` answers "is the process alive?" and never touches the LLM or the database. Kubernetes uses this for the `livenessProbe`: if it fails, restart the pod.

`/readyz` answers "is this pod ready to take traffic?". It checks four things:

1. The SQLite artifact store is writable.
2. The LLM key resolves a model (cheap probe — we construct the client, we don't make a chat call).
3. The search-tool credentials: `SERPAPI_API_KEY` is **required** (no SerpAPI = no real search = no useful workflow). `FIRECRAWL_API_KEY` is reported but not required — the scraper falls back to `httpx` if it's missing.
4. The LangGraph workflow compiled at startup.

If any required check fails, `/readyz` returns **503** and the orchestrator drains the pod from the service. Crucially, this **doesn't restart** the process — restarting won't fix a missing API key or an upstream LLM outage. That's the whole reason these are two endpoints.

In [2]:
r = requests.get(f"{BASE}/healthz")
print("/healthz  ->", r.status_code, r.json())

r = requests.get(f"{BASE}/readyz")
print(f"/readyz   -> {r.status_code}")
print(json.dumps(r.json(), indent=2))

/healthz  -> 200 {'status': 'ok'}
/readyz   -> 200
{
  "status": "ok",
  "checks": {
    "sqlite": {
      "ok": true,
      "path": "/app/runtime/artifacts.db"
    },
    "llm": {
      "ok": true,
      "detail": "ok",
      "model": "openai/gpt-5.4-mini"
    },
    "search_tools": {
      "ok": true,
      "detail": "ok",
      "serpapi": true,
      "firecrawl": true
    },
    "workflow": {
      "ok": true,
      "detail": "compiled"
    }
  }
}


### Live demo: take readiness red, then green

In another terminal, unset the key and restart the service:

```bash
unset OPENROUTER_API_KEY
PYTHONPATH=../../src uvicorn app.main:app --port 8090
```

Now `/healthz` is still `200` (the process is fine) but `/readyz` is `503` with `checks.llm.ok = false`. That's exactly what we want a load balancer to see: *don't send traffic here until the upstream config is fixed.* Set the key back and `/readyz` goes green without a restart.

## 3. Run the workflow

`POST /research` blocks until either the reflect LLM says `done=True` or the loop hits `max_iterations`. The body fields are:

| field | type | default | meaning |
|---|---|---|---|
| `topic` | str | required | The research question. |
| `max_iterations` | int (1-10) | `3` | Hard cap on plan → agent → reflect rounds. |

The response carries:

| field | meaning |
|---|---|
| `outcome` | `"complete"` if reflect said done, `"cap"` if we hit the iteration limit, `"error"` on graph failure. |
| `rounds` | How many plan/agent/reflect cycles ran. |
| `findings_count` | Total number of (sub-query → cited answer) findings across all rounds. |
| `artifact` | The full `Artifact` — `final_draft` markdown + per-round `provenance` (plan + findings + verdict). |

A typical run with the defaults takes 30-180 seconds — each plan step does one SerpAPI search and 1-3 Firecrawl scrapes, so latency scales with `rounds × steps × scrapes`. Cost is a few US cents on OpenRouter plus ~9 SerpAPI searches on the free tier.

In [3]:
t0 = time.time()
r = requests.post(
    f"{BASE}/research",
    json={
        "topic": "How does corrective RAG (CRAG) self-correct retrieval?",
        "max_iterations": 3,
    },
    timeout=600,
)
elapsed = time.time() - t0
r.raise_for_status()
result = r.json()
print(f"workflow took   : {elapsed:.1f}s")
print(f"outcome         : {result['outcome']}")
print(f"rounds          : {result['rounds']}")
print(f"findings_count  : {result['findings_count']}")
print(f"request_id      : {result['request_id']}")
print(f"artifact_id     : {result['artifact_id']}")

workflow took   : 81.6s
outcome         : complete
rounds          : 1
findings_count  : 4
request_id      : 9dba10040da84dcd8dfb289dacb1e8fb
artifact_id     : 662efc21-6382-4f6e-bd9d-0c2cd9a8f6dc


In [4]:
artifact = result["artifact"]

print("=" * 70)
print("PROVENANCE — one block per plan -> agent -> reflect round")
print("=" * 70)
for rec in artifact["provenance"]:
    plan = rec["plan"]
    verdict = rec["verdict"]
    print(f"\n  round {rec['round']}: "
          f"done={verdict['done']}  "
          f"steps={len(plan['steps'])}  "
          f"findings={len(rec['findings'])}")
    for i, step in enumerate(plan["steps"], 1):
        print(f"    Q{i}. {step['query'][:80]}")
    print(f"    verdict: {verdict['reasoning'][:120]}")
    if verdict["missing_questions"]:
        print(f"    follow-ups for next round:")
        for q in verdict["missing_questions"]:
            print(f"      - {q[:90]}")

print()
print("=" * 70)
print("FINAL DRAFT (first 2000 chars)")
print("=" * 70)
print(artifact["final_draft"][:2000])
print("..." if len(artifact["final_draft"]) > 2000 else "")

PROVENANCE — one block per plan -> agent -> reflect round

  round 1: done=True  steps=4  findings=4
    Q1. Corrective RAG CRAG paper self-correct retrieval mechanism
    Q2. CRAG confidence evaluator retrieval grading knowledge refinement search
    Q3. Corrective RAG retrieval transformation query rewriting web search augmentation
    Q4. CRAG vs standard RAG retrieval self correction benchmarks
    verdict: The findings already answer the core topic: CRAG self-corrects retrieval by inserting a lightweight retrieval evaluator 

FINAL DRAFT (first 2000 chars)
# How Corrective RAG (CRAG) Self-Corrects Retrieval

## Overview
Corrective Retrieval Augmented Generation (CRAG) is a plug-and-play extension to standard RAG that makes generation robust when retrieval is noisy, irrelevant, or incomplete by inserting an automatic self-correction layer between retrieval and generation [https://arxiv.org/abs/2401.15884]. Unlike conventional RAG, which indiscriminately consumes retrieved documents

## 4. Observability — metrics and trace replay

`/metrics` is the standard Prometheus text format. Three classes of metric matter most:

- `agent_workflow_runs_total{outcome="complete|cap|error"}` — terminal outcome counter
- `agent_reflect_rounds_total` — total plan→agent→reflect rounds across all runs (a proxy for cost; each round is a plan LLM call, N search+scrape passes per step, and a reflect LLM call)
- `agent_workflow_latency_seconds` — end-to-end latency histogram

Plus per-node timings under `agent_node_latency_seconds{node="plan|agent|reflect|artifact"}` if you want to find which stage is slow.

In [5]:
metrics_text = requests.get(f"{BASE}/metrics").text
interesting = [
    line for line in metrics_text.splitlines()
    if line.startswith(("agent_workflow_runs_total",
                        "agent_reflect_rounds_total",
                        "agent_workflow_latency_seconds_count",
                        "agent_workflow_latency_seconds_sum",
                        "agent_node_latency_seconds_count"))
    and not line.startswith("#")
]
print("\n".join(interesting))

agent_workflow_runs_total{outcome="complete"} 2.0
agent_reflect_rounds_total 2.0
agent_workflow_latency_seconds_count 2.0
agent_workflow_latency_seconds_sum 149.33852739999998
agent_node_latency_seconds_count{node="plan",status="ok"} 2.0
agent_node_latency_seconds_count{node="agent",status="ok"} 2.0
agent_node_latency_seconds_count{node="reflect",status="ok"} 2.0
agent_node_latency_seconds_count{node="artifact",status="ok"} 2.0


### Replay one request's log trail

This is the Segment 2 connection: every HTTP request gets a `request_id`, every log line written by the workflow carries it (threaded through a `ContextVar`), and `GET /trace?request_id=...` greps the JSON log file for just that id. So when a multi-step agent run does something weird, you have one ID to grep instead of a haystack.

In [6]:
trace = requests.get(f"{BASE}/trace", params={"request_id": result["request_id"]}).text

events = [json.loads(line) for line in trace.splitlines() if line]
print(f"{len(events)} log lines for request_id={result['request_id'][:8]}…\n")
for ev in events:
    label = ev.get("message", "")
    node = ev.get("node", "")
    extra = []
    for k in ("round", "step", "n_steps", "n_findings", "done",
              "draft_chars", "outcome", "status_class", "latency_s"):
        if k in ev:
            extra.append(f"{k}={ev[k]}")
    print(f"  [{ev.get('level','?'):<5}] {label:<20} node={node:<9} {' '.join(extra)}")

15 log lines for request_id=9dba1004…

  [INFO ] workflow.start       node=          
  [INFO ] node.start           node=plan      round=1
  [INFO ] node.end             node=plan      round=1 n_steps=4
  [INFO ] node.start           node=agent     round=1 n_steps=4
  [INFO ] agent.step           node=          round=1
  [INFO ] agent.step           node=          round=1
  [INFO ] agent.step           node=          round=1
  [INFO ] agent.step           node=          round=1
  [INFO ] node.end             node=agent     round=1
  [INFO ] node.start           node=reflect   round=1
  [INFO ] node.end             node=reflect   round=1 done=True
  [INFO ] loop_gate            node=          round=1 done=True
  [INFO ] node.start           node=artifact  n_findings=4 outcome=complete
  [INFO ] node.end             node=artifact  draft_chars=3638 outcome=complete
  [INFO ] http                 node=          status_class=2xx latency_s=81.6191


## 5. Force a failure mode — the iteration cap

The loop has two ways to terminate:

- **complete** — the reflect LLM looked at the accumulated findings and said `done=True`. The artifact is written from the full set of findings.
- **cap** — we hit `max_iterations` before reflect was satisfied. The artifact is *still written* (best-so-far from however many rounds we managed), and the metric label flips to `outcome="cap"`.

This second case is the cost lever. Without a cap a reflector that keeps asking follow-up questions will loop forever — your latency and bill go to infinity in lockstep. Below we set `max_iterations=1` on a broad, multi-faceted topic to make a cap likely: one round is rarely enough for reflect to declare victory on a question this open-ended.

In [7]:
r = requests.post(
    f"{BASE}/research",
    json={
        "topic": (
            "Survey every production tradeoff between liveness, readiness, "
            "and startup probes in Kubernetes — include real incident stories, "
            "tuning rules of thumb, and how service meshes interact with them."
        ),
        "max_iterations": 1,
    },
    timeout=600,
)
r.raise_for_status()
capped = r.json()
print(f"outcome        : {capped['outcome']}")
print(f"rounds         : {capped['rounds']}")
print(f"findings_count : {capped['findings_count']}")
assert capped["rounds"] == 1, "with max_iterations=1 we expect exactly one round"

print()
metrics_text = requests.get(f"{BASE}/metrics").text
for line in metrics_text.splitlines():
    if line.startswith("agent_workflow_runs_total{") and "#" not in line:
        print(line)

outcome        : cap
rounds         : 1
findings_count : 5

agent_workflow_runs_total{outcome="complete"} 2.0
agent_workflow_runs_total{outcome="cap"} 1.0


## 6. List the artifacts we've saved

`GET /artifacts` is the catalogue side — everything the workflow has persisted. The store is SQLite (`apps/agent_api/runtime/artifacts.db`). In a real deployment swap it for Postgres and add `?topic=` filtering if you need it.

In [8]:
listing = requests.get(f"{BASE}/artifacts", params={"limit": 10}).json()
print(f"{listing['total']} artifacts saved; showing {len(listing['items'])}:\n")
for item in listing["items"]:
    print(
        f"  {item['artifact_id'][:8]}  "
        f"rounds={item['rounds']}  "
        f"findings={item['findings_count']:>2}  "
        f"outcome={item['outcome']:<8}  "
        f"{item['topic'][:60]}"
    )

12 artifacts saved; showing 10:

  ba3098df  rounds=1  findings= 5  outcome=cap       Survey every production tradeoff between liveness, readiness
  662efc21  rounds=1  findings= 4  outcome=complete  How does corrective RAG (CRAG) self-correct retrieval?
  a413e68b  rounds=1  findings= 4  outcome=complete  Who is Sinan Ozdemir the AI author?
  aa707cb2  rounds=1  findings= 4  outcome=complete  How does corrective RAG (CRAG) self-correct retrieval?
  fdc1ad78  rounds=1  findings= 3  outcome=complete  Who is Sinan Ozdemir the AI author?
  9b97ae58  rounds=1  findings= 4  outcome=complete  How does corrective RAG (CRAG) self-correct retrieval?
  ae33b78a  rounds=1  findings= 4  outcome=complete  Sinan Ozdemir online deep dive: identity, notable work, publ
  51ab3f11  rounds=1  findings= 5  outcome=complete  Deep dive into Sinan Ozdemir online: identify who they are, 
  0b3e84bd  rounds=1  findings= 4  outcome=complete  How does corrective RAG (CRAG) self-correct retrieval?
  f7734bc5  rou

## 7. Install the MCP wrapper into Forge

We've shown the workflow runs and is observable. The last production beat is: **other agents need to call it.** The pattern is to ship an MCP server that wraps the API so any MCP-aware coding agent (Forge, Claude Code, Cursor, opencode) can install it.

The wrapper lives at [`apps/agent_api/mcp_server/research_workflow_server.py`](../../apps/agent_api/mcp_server/research_workflow_server.py). It exposes three tools — `research`, `get_artifact`, `list_artifacts` — each a thin HTTP call into the agent_api. (There used to be a fourth `health` tool, but every Forge built-in server was also exposing one and the LLM started getting confused by `health_2`, `health_3`, etc., so they're all gone. Use `GET /readyz` directly if you want a probe.)

Forge installs MCP servers as **JSON descriptors** — a small object naming the `command`, `args`, and `env` to spawn the server. (Forge used to also accept dropping a raw `.py` file; that's been removed. The descriptor is the only place to put `AGENT_API_BASE_URL`, and the only way to keep Forge from accepting executable uploads as a tool source.)

Build the descriptor and append a manifest entry:

In [9]:
REPO_ROOT = Path("../..").resolve()
FORGE_DIR = REPO_ROOT / ".forge"
FORGE_DIR.mkdir(exist_ok=True)
(FORGE_DIR / "mcp_servers").mkdir(exist_ok=True)

server_script = (APP_DIR / "mcp_server" / "research_workflow_server.py").resolve()
assert server_script.is_file(), server_script

descriptor = {
    "command": sys.executable,
    "args": [str(server_script)],
    "env": {
        "AGENT_API_BASE_URL": BASE,
        "AGENT_API_TIMEOUT_S": "600",
    },
}
descriptor_path = FORGE_DIR / "mcp_servers" / "research-workflow.json"
descriptor_path.write_text(json.dumps(descriptor, indent=2) + "\n")
print(f"descriptor -> {descriptor_path.relative_to(REPO_ROOT)}")

manifest_path = FORGE_DIR / "mcp_servers.json"
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
else:
    manifest = {"version": 1, "servers": []}

entry = {
    "name": "research-workflow",
    "kind": "json",
    "path": "research-workflow.json",
    "description": "Run a research workflow against the agent_api service.",
}
manifest["servers"] = [
    s for s in manifest["servers"] if s.get("name") != "research-workflow"
] + [entry]
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"manifest updated -> {manifest_path.relative_to(REPO_ROOT)}")
print(json.dumps(manifest, indent=2))

descriptor -> .forge/mcp_servers/research-workflow.json
manifest updated -> .forge/mcp_servers.json
{
  "version": 1,
  "servers": [
    {
      "name": "forge_descriptor",
      "kind": "json",
      "path": "forge_descriptor.json",
      "description": "",
      "extra": {
        "tools": [
          {
            "name": "research",
            "description": "Run a research workflow against the agent_api service.\n\n    Submits ``topic`` to the workflow, which loops research+judge until the\n    grade meets ``min_grade`` or ``max_iterations`` is hit. Returns the\n    artifact_id and a summary; use ``get_artifact`` to read the full draft.\n\n    This is a SYNCHRONOUS tool \u2014 it blocks until the workflow finishes\n    (typically 30-180 seconds). The caller's tool-call should expect a\n    long-running response.\n\n    Args:\n        topic: The research question or topic to brief.\n        min_grade: Minimum judge grade (0-5) required to stop early.\n        max_iterations: Cap o

Alternative — Forge install API: if `forge serve` is running on `127.0.0.1:6790`, POST the descriptor text and Forge will validate by spawning the server, listing its tools, then persisting it under `.forge/mcp_servers/` for you:

```python
import requests
contents = descriptor_path.read_text()
r = requests.post(
    "http://127.0.0.1:6790/api/mcp/install",
    json={"name": "research-workflow",
          "description": "agent_api research workflow",
          "contents": contents},
    timeout=30,
)
print(r.status_code, r.json())  # expect pending_restart: True
```

Either install path requires a `forge serve` restart so the new tools load — Forge does not hot-reload its MCP servers.

> **Two gotchas worth flagging,** both rooted in how Forge spawns the subprocess (via `asyncio.create_subprocess_exec`, **not** a shell):
>
> * `args[0]` must be an **absolute path**. Forge does not resolve relative paths from the descriptor's location.
> * `command` should be `sys.executable` (or `python3`), not bare `"python"`. On macOS `python` is usually a shell alias that doesn't exist as a real binary; `create_subprocess_exec` ignores aliases and you'll see `FileNotFoundError: 'python'`.

### Verify the MCP server speaks its protocol

Before we hand it to Forge, let's drive the server directly through stdio to make sure all three tools are registered. The MCP SDK's `ClientSession` is a clean way to do this from Python.

In [10]:
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


SERVER = str(APP_DIR / "mcp_server" / "research_workflow_server.py")

async def smoke_test_mcp():
    params = StdioServerParameters(
        command=sys.executable,
        args=[SERVER],
        env={**os.environ, "AGENT_API_BASE_URL": BASE},
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("tools exposed by research-workflow:")
            for t in tools.tools:
                print(f"  - {t.name}: {(t.description or '').strip().splitlines()[0]}")
            print()
            print("calling health()...")
            result = await session.call_tool("health", {})
            print(result.content[0].text[:400])

await smoke_test_mcp()

tools exposed by research-workflow:
  - research: Run a deep-research workflow against the agent_api service.
  - get_artifact: Fetch a previously-saved artifact and its full provenance.
  - list_artifacts: List saved artifacts, newest first.

calling health()...
Unknown tool: health


## 8. Drive the workflow from Forge

Restart `forge serve` (it doesn't hot-reload MCP servers). Then open a Forge chat session and ask:

> *Use the research-workflow MCP server to write a one-page brief on "observability for multi-step agent failures". Then call `get_artifact` to fetch the full draft.*

Forge will:

1. Call `research-workflow.research(topic=...)` — that's a stdio call into our MCP server, which makes an `httpx.post` to `http://localhost:8090/research`.
2. The agent_api runs the four-node workflow and persists the artifact.
3. Forge gets back the `artifact_id` + a truncated summary.
4. It then calls `research-workflow.get_artifact(artifact_id=...)` to pull the full draft and quote it back to you.

The cell below confirms the workflow ran by checking the metrics counter went up after the Forge turn. (You'd run the Forge chat in a separate terminal; the cell just verifies the upstream saw the request.)

In [11]:
def runs_total() -> int:
    text = requests.get(f"{BASE}/metrics").text
    n = 0
    for line in text.splitlines():
        if line.startswith("agent_workflow_runs_total{") and "#" not in line:
            n += int(float(line.rsplit(" ", 1)[-1]))
    return n

before = runs_total()
print(f"agent_workflow_runs_total (all outcomes) so far: {before}")
print()
print("Now drive Forge from another terminal:")
print("  cd apps/forge && python -m forge")
print("  > Use research-workflow to brief 'observability for multi-step agents'")
print()
print("Then re-run this cell — the counter should bump by at least 1.")

agent_workflow_runs_total (all outcomes) so far: 3

Now drive Forge from another terminal:
  cd apps/forge && python -m forge
  > Use research-workflow to brief 'observability for multi-step agents'

Then re-run this cell — the counter should bump by at least 1.


## 9. Cleanup

In [12]:
if proc is not None:
    proc.terminate()
    proc.wait(timeout=5)
    print(f"stopped uvicorn (pid={proc.pid})")
else:
    print("(nothing to stop — service was already running)")

(nothing to stop — service was already running)


## Exercises

1. **Distributed tracing.** Set `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY=...` in `apps/agent_api/.env`, restart, and rerun a request. Compare the LangSmith trace with `/trace?request_id=...`. Which one is easier to debug a silent judge approval with? When does each one fall short?

2. **Rate limiting.** The synchronous `/research` endpoint will happily exhaust your OpenRouter budget if hit in a loop. Add a per-IP rate-limit middleware (e.g. `slowapi`) and a request-budget metric (`agent_workflow_runs_dropped_total{reason="ratelimited"}`). Test it by spamming `requests.post` from a thread pool.

3. **Background + polling.** Right now `/research` blocks for the full loop. Swap it for two endpoints: `POST /jobs` (returns `{job_id}` immediately, schedules the workflow in a `BackgroundTasks` queue or Celery) and `GET /jobs/{id}` (returns running / succeeded / failed plus the artifact when done). Update the MCP `research` tool to call the new shape (poll the job endpoint until terminal). Which call style is better for Forge: synchronous-blocking or async-with-poll? Why?

4. **Cost tracking.** Wrap each LLM call in `CostTrackingLLM` (already in `src/shared/openrouter_llm.py`) and add a Prometheus histogram `agent_workflow_cost_usd`. Use it to plot the cost cliff: how does `max_iterations` correlate with the 95th-percentile cost per run?

5. **Probe both Forge AND Claude Desktop.** The MCP server is generic — same wrapper, two consumers. Drop the install JSON into Claude Desktop's MCP config and check that the same four tools show up. The point of MCP is exactly this: build once, plug into many hosts.